# 00 — Pipeline Logs Exploration

This notebook explores **all** logs, CSVs and reports produced by the Snakemake pipeline.
Files are mounted at `/pipeline-logs` from the host `./pipeline_logs` directory.

All cells are defensive: missing files are reported without raising exceptions.

## Pipeline execution order covered here

1. [Setup & directory overview](#1-setup)
2. [Metadata downloads & merging — HTML reports](#2-metadata)
3. [Database creation & validation](#3-database)
4. [Host resolution & download](#4-host-download)
5. [Host mapping creation](#5-host-mapping)
6. [Host FASTA QC & indexing](#6-host-qc)
7. [Host status report](#7-host-status)
8. [Phage & protein FASTA merging](#8-fasta-merge)
9. [Phage & protein FASTA indexing](#9-fasta-index)

---
<a id='1-setup'></a>
## 1 — Setup & Directory Overview

In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# ---------------------------------------------------------------------------
# Root path — matches PBI_LOGS_DIR env var inside the Docker container
# ---------------------------------------------------------------------------
logs_root = Path('/pipeline-logs')
logs_dir  = logs_root / 'logs'
csv_dir   = logs_root / 'csv'
rep_dir   = logs_root / 'reports'

print(f'Logs root : {logs_root}  (exists={logs_root.exists()})')
print(f'logs/     : {logs_dir.exists()}')
print(f'csv/      : {csv_dir.exists()}')
print(f'reports/  : {rep_dir.exists()}')

In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

MAX_LOG_LINES = 50  # max lines shown per text log


def show_log(path, max_lines=MAX_LOG_LINES):
    """Print the content of a text log file, or report if unavailable."""
    path = Path(path)
    print(f'\n=== {path} ===')
    if not path.exists():
        print('  (not available yet)')
        return
    if path.stat().st_size == 0:
        print('  (file exists but is empty)')
        return
    try:
        with path.open('r', encoding='utf-8', errors='replace') as fh:
            lines = fh.readlines()
        for line in lines[:max_lines]:
            print(line.rstrip('\n'))
        if len(lines) > max_lines:
            print(f'... ({len(lines) - max_lines} more lines — increase max_lines to see all)')
    except Exception as exc:
        print(f'  Could not read: {exc}')


def load_csv(path, label=None):
    """Load a CSV as a DataFrame and print a short summary."""
    path = Path(path)
    tag = label or path.name
    if not path.exists():
        print(f'[{tag}] not available yet.')
        return pd.DataFrame()
    if path.stat().st_size == 0:
        print(f'[{tag}] exists but is empty.')
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f'[{tag}] {len(df):,} rows × {len(df.columns)} columns')
        return df
    except Exception as exc:
        print(f'[{tag}] could not be loaded: {exc}')
        return pd.DataFrame()


def list_dir(d, indent=2):
    """Print a sorted listing of *d*."""
    d = Path(d)
    if not d.exists():
        print(' ' * indent + '(missing)')
        return
    items = sorted(d.iterdir(), key=lambda p: p.name.lower())
    if not items:
        print(' ' * indent + '(empty)')
        return
    for p in items:
        kind = 'DIR ' if p.is_dir() else 'FILE'
        size = '' if p.is_dir() else f'  {p.stat().st_size:>12,} B'
        print(' ' * indent + f'[{kind}] {p.name}{size}')

In [ ]:
# Full recursive listing of /pipeline-logs
print('── pipeline-logs/')
for sub in ('logs', 'csv', 'reports'):
    print(f'   ── {sub}/')
    list_dir(logs_root / sub, indent=6)

---
<a id='2-metadata'></a>
## 2 — Metadata Downloads & Merging

The pipeline downloads TSVs from PhageScope for each feature and merges them.
A ydata-profiling HTML report is generated per feature.  
No `.log` file is produced at this stage (downloads use `wget` directly via Snakemake shell rules).

> Files live in: `pipeline-logs/reports/{feature}_report.html`

In [ ]:
FEATURES = [
    'phage_metadata',
    'annotated_proteins_metadata',
    'transcription_terminator_metadata',
    'phage_trna_tmrna_metadata',
    'phage_anti_crispr_metadata',
    'phage_virulent_factor_metadata',
    'phage_transmembrane_protein_metadata',
    'antimicrobial_resistance_gene_metadata',
    'crispr_array_metadata',
]

print('Metadata report availability:')
for feat in FEATURES:
    p = rep_dir / f'{feat}_report.html'
    status = f'✅  {p.stat().st_size:,} B' if p.exists() else '❌  missing'
    print(f'  {feat:<50} {status}')

The HTML reports can be opened directly in the browser (they are mounted at `./pipeline_logs/reports/`).

---
<a id='3-database'></a>
## 3 — Database Creation & Validation

Rules: `create_duckdb` → `optimize_database` → `validate_database`  
The validation step produces an HTML report summarising table counts and integrity checks.

> File: `pipeline-logs/reports/database_validation.html`

In [ ]:
db_report = rep_dir / 'database_validation.html'
if db_report.exists():
    print(f'✅  database_validation.html  ({db_report.stat().st_size:,} B)')
    print('    Open in a browser or use the JupyterLab HTML viewer.')
else:
    print('❌  database_validation.html not available yet.')

---
<a id='4-host-download'></a>
## 4 — Host Resolution & Download

Rule: `download_host_genomes` (script: `download_host_genomes_robust.py`)

This step:
1. Parses each phage's `Host` field into candidate tokens.
2. Resolves tokens to NCBI assembly accessions via `AssemblyResolver`.
3. Downloads genome FASTA files via FTP.

**Log files**
| File | Description |
|------|-------------|
| `logs/host_download.log` | Full execution log (resolver + downloader) |
| `logs/host_download_failures.log` | Per-accession failures only |

**CSV outputs** (in `csv/`)
| File | Description |
|------|-------------|
| `phage_host_candidates.csv` | One row per (Phage_ID, Host_Token) |
| `phage_host_assemblies.csv` | Resolved assembly links with confidence scoring |
| `assembly_metadata.csv` | Download status per assembly |
| `host_metadata.csv` | High-level host metadata |
| `phage_host_links.csv` | Final phage → assembly links |
| `host_token_resolution_cache.json` | Persistent token→assembly cache |

In [ ]:
# ── Text logs ──────────────────────────────────────────────────────────────
show_log(logs_dir / 'host_download.log')
show_log(logs_dir / 'host_download_failures.log')

In [ ]:
# ── phage_host_candidates.csv ──────────────────────────────────────────────
candidates = load_csv(csv_dir / 'phage_host_candidates.csv')
if not candidates.empty:
    print('\nColumns:', candidates.columns.tolist())
    display(candidates.head(10))
    
    print('\n--- Token type distribution ---')
    if 'Token_Type' in candidates.columns:
        display(candidates['Token_Type'].value_counts().rename('count'))
    
    print(f'\nUnique phages : {candidates["Phage_ID"].nunique():,}')
    print(f'Total tokens  : {len(candidates):,}')

In [ ]:
# ── phage_host_assemblies.csv ──────────────────────────────────────────────
assemblies = load_csv(csv_dir / 'phage_host_assemblies.csv')
if not assemblies.empty:
    print('\nColumns:', assemblies.columns.tolist())
    display(assemblies.head(10))

    if 'Resolution_Source' in assemblies.columns:
        print('\n--- Resolution source distribution ---')
        display(assemblies['Resolution_Source'].value_counts().rename('count'))

    if 'Ambiguous' in assemblies.columns:
        n_amb = assemblies['Ambiguous'].sum()
        print(f'\nAmbiguous resolutions : {n_amb:,} / {len(assemblies):,}')

    if 'Assembly_Level' in assemblies.columns:
        print('\n--- Assembly level distribution ---')
        display(assemblies['Assembly_Level'].value_counts().rename('count'))

In [ ]:
# ── assembly_metadata.csv ─────────────────────────────────────────────────
asm_meta = load_csv(csv_dir / 'assembly_metadata.csv')
if not asm_meta.empty:
    print('\nColumns:', asm_meta.columns.tolist())
    display(asm_meta.head(10))

    if 'Download_Status' in asm_meta.columns:
        print('\n--- Download status distribution ---')
        display(asm_meta['Download_Status'].value_counts().rename('count'))

    if 'Metadata_Only' in asm_meta.columns:
        print(f'\nMetadata-only (no FASTA downloaded): {asm_meta["Metadata_Only"].sum():,}')

In [ ]:
# ── host_metadata.csv ────────────────────────────────────────────────────
host_meta = load_csv(csv_dir / 'host_metadata.csv')
if not host_meta.empty:
    print('\nColumns:', host_meta.columns.tolist())
    display(host_meta.head(10))

In [ ]:
# ── phage_host_links.csv ─────────────────────────────────────────────────
links = load_csv(csv_dir / 'phage_host_links.csv')
if not links.empty:
    print('\nColumns:', links.columns.tolist())
    display(links.head(10))
    print(f'\nUnique phages with a host link : {links["Phage_ID"].nunique():,}')

In [ ]:
# ── host_token_resolution_cache.json ────────────────────────────────────
cache_path = csv_dir / 'host_token_resolution_cache.json'
if cache_path.exists():
    with cache_path.open('r') as fh:
        cache = json.load(fh)
    n_tokens = len(cache)
    n_resolved = sum(1 for v in cache.values() if v)
    n_empty = n_tokens - n_resolved
    print(f'Token resolution cache: {n_tokens:,} tokens total')
    print(f'  Resolved (≥1 assembly) : {n_resolved:,}')
    print(f'  Empty (no assembly)    : {n_empty:,}')
else:
    print('host_token_resolution_cache.json not available yet.')

---
<a id='5-host-mapping'></a>
## 5 — Host Mapping Creation

Rule: `create_host_mapping` (script: `create_host_mapping.py`)

Builds a `Host_ID → FASTA path` JSON from the downloaded assemblies and any private host data.

> Log file: `logs/create_host_mapping.log`

In [ ]:
show_log(logs_dir / 'create_host_mapping.log')

---
<a id='6-host-qc'></a>
## 6 — Host FASTA QC & Indexing

Rule: `index_individual_host_sequences` (script: `index_individual_hosts.py`)

QC procedure per host FASTA (non-destructive):
1. **Header audit**: files with duplicate sequence identifiers are *rejected* (not indexed).
2. **Sequence content audit**: identical sequences trigger a warning but the file is still indexed.

**Outputs**
| File | Description |
|------|-------------|
| `logs/index_individual_host_sequences.log` | Step-by-step execution log |
| `logs/host_fasta_qc.csv` | Per-file QC results (DataFrame-loadable) |

In [ ]:
show_log(logs_dir / 'index_individual_host_sequences.log')

In [ ]:
# ── host_fasta_qc.csv ─────────────────────────────────────────────────────
qc = load_csv(logs_dir / 'host_fasta_qc.csv')
if not qc.empty:
    print('\nColumns:', qc.columns.tolist())
    display(qc.head(10))

    if 'index_status' in qc.columns:
        print('\n--- Index status distribution ---')
        display(qc['index_status'].value_counts().rename('count'))

    if 'header_qc_status' in qc.columns:
        print('\n--- Header QC status distribution ---')
        display(qc['header_qc_status'].value_counts().rename('count'))

    if 'seq_qc_status' in qc.columns:
        print('\n--- Sequence QC status distribution ---')
        display(qc['seq_qc_status'].value_counts().rename('count'))

    # Rejected files (duplicate headers)
    if 'index_status' in qc.columns:
        rejected = qc[qc['index_status'] == 'rejected_duplicate_headers']
        if not rejected.empty:
            print(f'\n--- Rejected files ({len(rejected):,}) ---')
            display(rejected[['host_id', 'n_duplicate_headers',
                               'duplicate_header_examples', 'error_message']].head(20))

    # Files with duplicate sequences (warning)
    if 'n_identical_seq_groups' in qc.columns:
        dup_seq = qc[qc['n_identical_seq_groups'].fillna(0) > 0]
        if not dup_seq.empty:
            print(f'\n--- Files with duplicate sequences ({len(dup_seq):,}) ---')
            display(dup_seq[['host_id', 'n_identical_seq_groups',
                              'identical_seq_group_examples']].head(20))

---
<a id='7-host-status'></a>
## 7 — Host Status Report

Rule: `create_host_status_report` (script: `create_host_status_report.py`)

Joins the four upstream tables (candidates, assemblies, assembly_metadata, QC log)
into a single per-`(Phage_ID, Host_Token)` summary.

**Outputs**
| File | Description |
|------|-------------|
| `logs/create_host_status_report.log` | Step-by-step log |
| `reports/host_status_report.csv` | Combined status table |

In [ ]:
show_log(logs_dir / 'create_host_status_report.log')

In [ ]:
# ── host_status_report.csv ───────────────────────────────────────────────
status_report = load_csv(rep_dir / 'host_status_report.csv')
if not status_report.empty:
    print('\nColumns:', status_report.columns.tolist())
    display(status_report.head(10))

    # Key summary metrics
    n_phages  = status_report['Phage_ID'].nunique()
    print(f'\nTotal unique phages            : {n_phages:>8,}')

    if 'Resolution_Status' in status_report.columns:
        n_res = status_report.loc[
            status_report['Resolution_Status'] == 'resolved', 'Phage_ID'
        ].nunique()
        print(f'Phages with ≥1 resolved host   : {n_res:>8,}')
        print(f'Phages with no resolved host   : {n_phages - n_res:>8,}')

        print('\n--- Resolution status distribution ---')
        display(status_report['Resolution_Status'].value_counts().rename('rows'))

    if 'Download_Status' in status_report.columns:
        print('\n--- Download status distribution ---')
        display(status_report['Download_Status'].value_counts(dropna=False).rename('rows'))

    if 'index_status' in status_report.columns:
        print('\n--- Index status distribution ---')
        display(status_report['index_status'].value_counts(dropna=False).rename('rows'))

    # Assemblies with failed downloads
    if 'Download_Status' in status_report.columns:
        failures = status_report[
            status_report['Download_Status'].notna() &
            (status_report['Download_Status'] != 'success')
        ].drop_duplicates('Assembly_Accession')
        if not failures.empty:
            cols = [c for c in ['Assembly_Accession', 'Organism_Name', 'Strain',
                                 'Download_Status'] if c in failures.columns]
            print(f'\n--- Download failures ({len(failures):,} assemblies) ---')
            display(failures[cols].head(20))

---
<a id='8-fasta-merge'></a>
## 8 — Phage & Protein FASTA Merging

Rules: `merge_phage_fasta` and `merge_protein_fasta` (script: `merge_fasta.py`)

Merges per-source FASTA files into a single combined FASTA for phages and proteins.

> Log files: `logs/merge_phage_fasta.log`, `logs/merge_protein_fasta.log`

In [ ]:
show_log(logs_dir / 'merge_phage_fasta.log')

In [ ]:
show_log(logs_dir / 'merge_protein_fasta.log')

---
<a id='9-fasta-index'></a>
## 9 — Phage & Protein FASTA Indexing

Rules: `index_phage_sequences` and `index_protein_sequences` (script: `index_sequences.py`)

Each step:
1. Normalises and deduplicates sequences in place.
2. Creates a pyfaidx `.fai` index.
3. Writes a duplicate analysis HTML report to `reports/`.

> Log files: `logs/index_phage_sequences.log`, `logs/index_protein_sequences.log`  
> HTML reports: `reports/{fasta_name}_duplicates.html`

In [ ]:
show_log(logs_dir / 'index_phage_sequences.log')

In [ ]:
show_log(logs_dir / 'index_protein_sequences.log')

In [ ]:
# Duplicate analysis HTML reports (generated by index_sequences.py)
dup_reports = sorted(rep_dir.glob('*_duplicates.html')) if rep_dir.exists() else []
if dup_reports:
    print('Duplicate analysis reports:')
    for p in dup_reports:
        print(f'  ✅  {p.name}  ({p.stat().st_size:,} B)')
else:
    print('No duplicate analysis reports found yet.')